In [1]:
from perflib.notebook import use_full_width, activate_autoreload
use_full_width()
activate_autoreload()
%matplotlib inline

import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.ticker import FuncFormatter
import matplotlib.cm as cm

import numpy as np
from bladf import Database, Label, utils
from bladf.utils import plot_label

import pandas as pd
from perflib.utils import atmos_isa

import os
import shutil
import json
from pathlib import Path

In [2]:
# USER INPUT 
# USER INPUT 
SCRIPT_DIRECTORY = "/projects/fms_exp/uerrxn0d/SCRIPT/VULCAN_VAL/A321-276B/INPUT_HORAIRE_MCL/"
SETTINGS_FILE = "F1.13.0_Settings_A321_276B_VN10_C00_Horaire_mcl"
INPUT_FILE = "inputs_A321_276B_VN10_C00.json"
OUTPUT_DIRECTORY = "/projects/fms_exp/uerrxn0d/SCRIPT/VULCAN_VAL/A321-276B/OUTPUT_HORAIRE_MCL/"
OUTPUT_CSV = "details_results.csv"

GW_min = 50000
GW_max = 100000
GW_step = 5000

DISA_min = -40
DISA_max = 40
DISA_step = 5

STEP = 1000
ZP_min = 20000


In [3]:
# CREATE RANGES THAT WILL BE USED
GW_range = np.arange(GW_min, GW_max + GW_step, GW_step)
DISA_range = np.arange(DISA_min, DISA_max + DISA_step, DISA_step)

GW_range, DISA_range

(array([ 50000,  55000,  60000,  65000,  70000,  75000,  80000,  85000,
         90000,  95000, 100000]),
 array([-40, -35, -30, -25, -20, -15, -10,  -5,   0,   5,  10,  15,  20,
         25,  30,  35,  40]))

In [4]:
def horaire_alt(
    df: pd.DataFrame,
    HRECMAXALT,
    hcermaxalt: float    
)-> pd.DataFrame:
    
    recmaxalt_values = []

    for _, row in df.iterrows():
        disa = row["DISA_K"]
        gw = row["GW_KG"]
        recmaxalt = min(
            hcermaxalt,
            HRECMAXALT.interpolate([disa, gw], method="point")
        )
        recmaxalt_values.append(recmaxalt)
        
    df["ZP_FT"] = recmaxalt_values
    
    return df

In [5]:
from itertools import product

combinations = list(product(GW_range, DISA_range))

df = pd.DataFrame(combinations, columns=["GW_KG", "DISA_K"])

# Compute Corresponding VMAX and Altitudes Automatically
FMS_model_bladf = Database("my_bl", "BL", "/projects/fms_exp/ng742da/WORK/nFMS/PDB/PDB_Audit_2026/A321-276B-VN10-C00/A321-276B-VN10-C00")
HRECMAXALT = FMS_model_bladf.get_label('HRECMAXALT')
HCERMAXALT = FMS_model_bladf.get_label('HCERMAXALT')
hcermaxalt = HCERMAXALT.values[0]

df = horaire_alt(
    df,
    HRECMAXALT,
    hcermaxalt = hcermaxalt
)

# Create expanded dataframe
df_expanded = (
    df
    .assign(ZP_FT=lambda x: x["ZP_FT"].apply(
        lambda max_alt: np.arange(ZP_min, max_alt + STEP, STEP)
    ))
    .explode("ZP_FT")
    .reset_index(drop=True)
)
df_expanded

,GW_KG,DISA_K,ZP_FT
0,50000,-40,20000.0
1,50000,-40,21000.0
2,50000,-40,22000.0
3,50000,-40,23000.0
4,50000,-40,24000.0
...,...,...,...
3344,100000,35,25000.0
3345,100000,40,20000.0
3346,100000,40,21000.0
3347,100000,40,22000.0


In [6]:
# Compute the maximumum cruise speed from the above dataset
def horaire_max_speed(
    df: pd.DataFrame,
    HMAXCL,
    HMAXBUF,
    atmos_isa,
    vmo: float,
    mmo: float
) -> pd.DataFrame: 

    vmax_mach_values = []
    vmax_buf_values = []

    for _, row in df.iterrows():
        disa = row["DISA_K"]
        gw = row["GW_KG"]
        zp = row["ZP_FT"]


        # --- VMAX MACH ---
        vmaxcl_mach = HMAXCL.interpolate([disa, gw, zp], method="point", errors='ignore',
                                interpolate=['linear', 'linear', 'linear'],
                                extrapolate=['linear', 'linear', 'linear'])
        vmaxbuf_mach = HMAXBUF.interpolate([(gw / 1000), zp], method="point")
        
        vmo_mach = atmos_isa.mach_cas(vmo, zp)
        
        vmax_mach_values.append(vmaxcl_mach)
        vmax_buf_values.append(vmaxbuf_mach)

    # Add the maximum cruise speed data frame
    
    df["V_MAX_MACH_NONE"] = vmax_mach_values

    df = df[
    (df["V_MAX_MACH_NONE"] <= mmo) & 
    (df["V_MAX_MACH_NONE"] <= vmax_buf_values)&
    (df["V_MAX_MACH_NONE"] <= vmo_mach)
    ].copy()
    
    return df

In [7]:
HMAXCL = FMS_model_bladf.get_label('HMAXCL')
HMAXBUF = FMS_model_bladf.get_label('HMAXBUF')
HVMAX = FMS_model_bladf.get_label('HVMAX')
vmo = HVMAX.values[0, 0]
mmo = HVMAX.values[1, 0]
df = horaire_max_speed(
    df_expanded,
    HMAXCL,
    HMAXBUF,
    atmos_isa = atmos_isa,
    vmo = vmo,
    mmo = mmo 
)
df

,GW_KG,DISA_K,ZP_FT,V_MAX_MACH_NONE
356,50000,40,40000.0,0.788000
692,55000,35,40000.0,0.783500
712,55000,40,39000.0,0.784000
1047,60000,35,39000.0,0.774500
1065,60000,40,37000.0,0.795000
...,...,...,...,...
3344,100000,35,25000.0,0.758833
3345,100000,40,20000.0,0.752000
3346,100000,40,21000.0,0.748400
3347,100000,40,22000.0,0.744800


In [8]:
# --- define your limits ---
DISA_limit = 40
GW_limit = 100000

# Count before filtering
print("Points before filtering:", len(df))

# Filter out the unwanted corner of the envelope
df = df[~((df["DISA_K"] >= DISA_limit) & (df["GW_KG"] >= GW_limit))]

# Reset index (clean dataframe)
df = df.reset_index(drop=True)

print("Points after filtering:", len(df))
df

Points before filtering: 155
Points after filtering: 151


,GW_KG,DISA_K,ZP_FT,V_MAX_MACH_NONE
0,50000,40,40000.0,0.788000
1,55000,35,40000.0,0.783500
2,55000,40,39000.0,0.784000
3,60000,35,39000.0,0.774500
4,60000,40,37000.0,0.795000
...,...,...,...,...
146,100000,35,21000.0,0.775300
147,100000,35,22000.0,0.773100
148,100000,35,23000.0,0.769833
149,100000,35,24000.0,0.765500


In [9]:
def dataframe_to_doe(df):
    doe = {}
    
    for column in df.columns:
        doe[column] = {
            "level": 1,
            "vector": df[column].tolist()
        }
        
    return doe

In [10]:
json_data = {
    "horaire_mcl": {
        "activated": True,
        "design_of_experiments": dataframe_to_doe(df),
        "settings": {
            "additive_margin_on_the_minimum_mach_envelope": 0,
            "additive_margin_on_the_maximum_mach_envelope": 0,
            "additive_margin_on_the_ceiling": 0,
            "max_it_failure_rate": 12
        },
        "checks": [
            {
                "activated": True,
                "title": "Check RFNIR",
                "type": "fixed_tolerances",
                "perfo_item": "TSP_RFNIR",
                "tolerance_absolute_min": -0.75,
                "tolerance_absolute_max": 0.75,
                "tolerance_relative_min": None,
                "tolerance_relative_max": None
            },
            {
                "activated": True,
                "title": "Check RTSP",
                "type": "fixed_tolerances",
                "perfo_item": "TSP_RTSP",
                "tolerance_absolute_min": -0.75,
                "tolerance_absolute_max": 0.75,
                "tolerance_relative_min": None,
                "tolerance_relative_max": None
            }
        ]
    }
}

In [11]:
output_file = Path(SETTINGS_FILE+".json")

with open(output_file, "w") as f:
    json.dump(json_data, f, indent=4)

print("JSON generated ✔")

JSON generated ✔
